In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

In [ ]:
pfc1 <- readRDS("data/processed/batch_integrated_file")

In [ ]:
## 可视化质量

In [ ]:
vln_plot <- VlnPlot(pfc1, features = c("nCount_RNA", "nFeature_RNA"), group.by = "seurat_clusters", ncol = 2, pt.size=0)
DimPlot(pfc1, reduction = "umap", group.by = "seurat_clusters", label = TRUE)

In [ ]:
## 大类经典marker注释

In [ ]:
p1 <- FeaturePlot(pfc1, features = c("SATB2", "NEFM","Slit3","SLC17A6","SLC17A7"), reduction = "umap")
ggsave("data/processed/batch_integrated_file", plot = p1, width = 10, height = 15)

p2 <- FeaturePlot(pfc1, features = c("GAD1.1","GAD2"), reduction = "umap")#"BTBD11"
ggsave("data/processed/batch_integrated_file", plot = p2, width = 10, height = 5)

p3 <- FeaturePlot(pfc1, features = c("MBP", "OPALIN","MOG","MOBP","PLP1","BCAS1"), reduction = "umap")
ggsave("data/processed/batch_integrated_file", plot = p3, width = 10, height = 15)

p4 <- FeaturePlot(pfc1, features = c("SLC1A2","SLC1A3","GFAP"), reduction = "umap")
ggsave("data/processed/batch_integrated_file", plot = p4, width = 10, height = 10)

p5 <- FeaturePlot(pfc1, features = c("VCAN","TNR","PDGFRA","SOX6","PCDH15"), reduction = "umap")
ggsave("data/processed/batch_integrated_file", plot = p5, width = 10, height = 15)

p6 <- FeaturePlot(pfc1, features = c("IGFBP7","FLT1","PDGFRB","CLDN5"), reduction = "umap")
ggsave("data/processed/batch_integrated_file", plot = p6, width = 10, height = 10)

In [ ]:
# 查看 seurat_clusters 的唯一值
unique_clusters <- unique(pfc1@meta.data$seurat_clusters)
print(unique_clusters)

# 手动创建一个簇编号到注释标签的映射表
cluster_annotation <- c(
  "0" = "Oligo",
  "1" = "Astro",
  "2" = "Ex1",
  "3" = "Endo",
  "4" = "Opc",
  "5" = "Ex2",
  "6" = "In",
  "7" = "Immune",
  "8" = "Other",
  "9" = "In",
  "10" = "Oligo/Opc",
  "11" = "Astro",
  "12" = "Ep",
  "13" = "Ex3",
  "14" = "Ex4",
  "15" = "Astro",
  "16" = "Fibroblast"
)

# 根据 seurat_clusters 的值，为每个细胞分配注释标签
pfc1@meta.data$annotation <- cluster_annotation[as.character(pfc1@meta.data$seurat_clusters)]

# 查看结果
head(pfc1@meta.data[c("seurat_clusters", "annotation")])

In [ ]:
# 筛选去除“Other”细胞
pfc1_filtered <- subset(pfc1, subset = annotation != "Other")
DimPlot(pfc1_filtered, reduction = "umap", group.by = "annotation")

In [ ]:
## 提取兴奋性神经元

In [ ]:
Idents(pfc1) <- pfc1@meta.data$annotation
excit_neurons <- subset(pfc1, idents = c("Ex1", "Ex2", "Ex3", "Ex4"))
DimPlot(excit_neurons, reduction = "umap", group.by = "annotation")

In [ ]:
## harmony整合鲸豚皮层所有sc

In [ ]:
v112 <- readRDS("data/processed/cortex_analysis_file")
pfc10 <- readRDS("data/processed/cortex_analysis_file")
pfc1 <- readRDS("data/processed/cortex_analysis_file")
v117 <- readRDS("data/processed/cortex_analysis_file")
pfc33 <- readRDS("data/processed/cortex_analysis_file")

In [ ]:
## 添加物种和脑区标签

In [ ]:
pfc10@meta.data$species <- rep("NA", nrow(pfc10@meta.data))
pfc1@meta.data$species <- rep("NA", nrow(pfc1@meta.data))
pfc33@meta.data$species <- rep("SA", nrow(pfc33@meta.data))
v112@meta.data$species <- rep("NA", nrow(v112@meta.data))
v117@meta.data$species <- rep("SA", nrow(v117@meta.data))

pfc10@meta.data$region <- rep("PFC", nrow(pfc10@meta.data))
pfc1@meta.data$region <- rep("PFC", nrow(pfc1@meta.data))
pfc33@meta.data$region <- rep("PFC", nrow(pfc33@meta.data))
v112@meta.data$region <- rep("V1", nrow(v112@meta.data))
v117@meta.data$region <- rep("V1", nrow(v117@meta.data))

In [ ]:
seurat_list <- list(pfc10, pfc1, pfc33, v112, v117)
merged_seurat <- merge(seurat_list[[1]], y = seurat_list[-1], add.cell.ids = c("pfc10", "pfc1", "pfc33", "v112", "v117"))

In [ ]:
# 数据标准化和降维
merged_seurat <- NormalizeData(merged_seurat)
merged_seurat <- FindVariableFeatures(merged_seurat)
merged_seurat <- ScaleData(merged_seurat)
merged_seurat <- RunPCA(merged_seurat, features = VariableFeatures(object = merged_seurat))
merged_seurat <- RunHarmony(merged_seurat, group.by = "orig.ident")

In [ ]:
# 合并批次标签和物种标签
merged_seurat <- AddMetaData(merged_seurat, metadata = paste(merged_seurat$orig.ident, merged_seurat$species, sep = "_"), col.name = "batch_species")
# 使用新的标签进行批次校正
merged_seurat <- RunHarmony(merged_seurat, group.by = "batch_species")
merged_seurat <- RunUMAP(merged_seurat, reduction = "harmony", dims = 1:20)

In [ ]:
# 绘制UMAP图
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5)
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "orig.ident")
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "species")
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "region")
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "annotation")

In [ ]:
saveRDS(merged_seurat, file = "data/processed/cortex_analysis_file")

In [ ]:
## harmony整合所有兴奋性神经元

In [ ]:
v112 <- readRDS("data/processed/excitatory_neuron_file")
pfc10 <- readRDS("data/processed/excitatory_neuron_file")
pfc1 <- readRDS("data/processed/excitatory_neuron_file")
v117 <- readRDS("data/processed/excitatory_neuron_file")
pfc33 <- readRDS("data/processed/excitatory_neuron_file")

In [ ]:
pfc10@meta.data$species <- rep("NA", nrow(pfc10@meta.data))
pfc1@meta.data$species <- rep("NA", nrow(pfc1@meta.data))
pfc33@meta.data$species <- rep("SA", nrow(pfc33@meta.data))
v112@meta.data$species <- rep("NA", nrow(v112@meta.data))
v117@meta.data$species <- rep("SA", nrow(v117@meta.data))

pfc10@meta.data$region <- rep("PFC", nrow(pfc10@meta.data))
pfc1@meta.data$region <- rep("PFC", nrow(pfc1@meta.data))
pfc33@meta.data$region <- rep("PFC", nrow(pfc33@meta.data))
v112@meta.data$region <- rep("V1", nrow(v112@meta.data))
v117@meta.data$region <- rep("V1", nrow(v117@meta.data))

In [ ]:
seurat_list <- list(pfc10, pfc1, pfc33, v112, v117)
merged_seurat <- merge(seurat_list[[1]], y = seurat_list[-1], add.cell.ids = c("pfc10", "pfc1", "pfc33", "v112", "v117"))

In [ ]:
# 数据标准化和降维
merged_seurat <- NormalizeData(merged_seurat)
merged_seurat <- FindVariableFeatures(merged_seurat)
merged_seurat <- ScaleData(merged_seurat)
merged_seurat <- RunPCA(merged_seurat, features = VariableFeatures(object = merged_seurat))
merged_seurat <- RunHarmony(merged_seurat, group.by = "orig.ident")

In [ ]:
# 使用Harmony降维结果重新聚类
merged_seurat <- RunUMAP(merged_seurat, reduction = "harmony", dims = 1:20)
merged_seurat <- FindNeighbors(merged_seurat, reduction = "harmony", dims = 1:20)
merged_seurat <- FindClusters(merged_seurat, resolution = 0.5)

In [ ]:
# 使用Harmony降维结果重新聚类
merged_seurat <- RunUMAP(merged_seurat, reduction = "harmony", dims = 1:20)
merged_seurat <- FindNeighbors(merged_seurat, reduction = "harmony", dims = 1:20)
merged_seurat <- FindClusters(merged_seurat, resolution = 0.5)

In [ ]:
# 绘制UMAP图
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5)
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "orig.ident")
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "species")
DimPlot(merged_seurat, reduction = "umap", label = TRUE, pt.size = 0.5, group.by = "region")

In [ ]:
saveRDS(merged_seurat, file = "data/processed/excitatory_neuron_file")

In [ ]:
## 兴奋性神经元重新注释

In [ ]:
dataha <- readRDS("data/processed/excitatory_neuron_file")
dataha

In [ ]:
# 提取原始的seurat_clusters
original_clusters <- dataha$seurat_clusters

# 创建一个映射表，将原始标签映射到新的标签
new_cluster_map <- c(
  "5" = "Other", "18" = "Other",
  "0" = "EX_ITGA4_TPBG", 
  "1" = "EX_HTR7_GNAL", "2" = "EX_HTR7_GNAL", 
  "3" = "EX_FOXP2", "16" = "EX_FOXP2", "9" = "EX_FOXP2",
  "4" = "EX_NTNG1", 
  "6" = "EX_ERG", 
  "7" = "EX_C1QL3",
  "8" = "EX_env_TMEM144", 
  "10" = "EX_SLC1A3_FGFR3",
  "11" = "EX_Ntng2", 
  "12" = "EX_ADRA1A_Vat1l", 
  "13" = "EX_BDNF", 
  "14" = "EX_TSHZ2_VWC2L", 
  "15" = "EX_Nxph4_CPLX3",  
  "17" = "EX_IFITM3_CLDN5"
)

# 使用映射表将原始标签映射到新的注释标签
new_clusters <- new_cluster_map[as.character(original_clusters)]

# 检查是否有未映射的值（即原始标签不在映射表中）
if (any(is.na(new_clusters))) {
  warning("Some original cluster labels are not mapped to new labels. These will be set to NA.")
}

# 将新的注释标签添加到dataha的meta.data中
dataha <- AddMetaData(dataha, metadata = new_clusters, col.name = "annotation")

# 查看结果
head(dataha@meta.data)

In [ ]:
saveRDS(dataha, file = "data/processed/excitatory_neuron_file")